# Phase 5: Statistical Analysis

Chi-square (risk factor vs discomfort), Mann-Whitney / Kruskal-Wallis, logistic regression with odds ratios. Tables saved to outputs/tables/.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
df     = pd.read_csv(ROOT / 'data' / 'processed' / 'model_ready.csv')
risk   = pd.read_csv(ROOT / 'data' / 'processed' / 'risk_profile.csv')
TABLES = ROOT / 'outputs' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)

print('model_ready :', df.shape)
print('risk_profile:', risk.shape)

model_ready : (182, 91)
risk_profile: (182, 14)


## Step 1 — Build the analysis frame

In [2]:
# Risk labels + discomfort outcome side by side, aligned on row index
analysis = risk.copy()
analysis['discomfort'] = df['discomfort'].values

print('discomfort distribution:', analysis['discomfort'].value_counts().to_dict())
analysis.head()

discomfort distribution: {1: 154, 0: 28}


,rider_id,timestamp,force_risk,force_risk_code,repetition_risk,repetition_risk_code,duration_risk,duration_risk_code,vibration_risk,vibration_risk_code,contact_stress_risk,contact_stress_risk_code,posture_risk,posture_risk_code,discomfort
0,0,3/3/2026 0:41:28,Low,0,Medium,1,Low,0,Low,0,Low,0,Medium,1,0
1,1,3/3/2026 0:44:51,Low,0,Low,0,Medium,1,Low,0,High,2,High,2,1
2,2,3/3/2026 0:47:05,Low,0,Medium,1,High,2,High,2,Medium,1,High,2,1
3,3,3/3/2026 0:49:52,Low,0,Medium,1,Low,0,Low,0,Medium,1,Medium,1,1
4,4,3/3/2026 0:52:25,Low,0,Low,0,Medium,1,Medium,1,High,2,High,2,1


## Step 2 — Chi-square: each risk factor vs discomfort

In [3]:
risk_factors = ['force_risk', 'repetition_risk', 'duration_risk',
                'vibration_risk', 'contact_stress_risk', 'posture_risk']

chi_results = []
for f in risk_factors:
    ct = pd.crosstab(analysis[f], analysis['discomfort'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    chi_results.append({
        'risk_factor': f.replace('_risk', ''),
        'chi2':        round(chi2, 3),
        'dof':         dof,
        'p_value':     round(p, 4),
        'significant': p < 0.05
    })

chi_df = pd.DataFrame(chi_results).sort_values('p_value')
chi_df.to_csv(TABLES / 'chi_square.csv', index=False)
chi_df

,risk_factor,chi2,dof,p_value,significant
5,posture,45.665,1,0.0000,True
0,force,6.718,2,0.0348,True
2,duration,0.623,2,0.7325,False
3,vibration,0.600,2,0.7408,False
4,contact_stress,0.544,2,0.7617,False
1,repetition,0.415,2,0.8128,False


## Step 3 — Mann-Whitney U: continuous predictors split by discomfort

In [4]:
numeric_features = ['workload_score', 'fatigue_score', 'force_exertion',
                    'vibration_index', 'work_hours_num', 'deliveries_num',
                    'work_days_num', 'rest_break_num',
                    'age_ord', 'job_duration_ord', 'income_ord', 'education_ord']

mw_results = []
for feat in numeric_features:
    grp0 = df.loc[df['discomfort'] == 0, feat]
    grp1 = df.loc[df['discomfort'] == 1, feat]
    u, p = stats.mannwhitneyu(grp0, grp1, alternative='two-sided')
    mw_results.append({
        'feature':              feat,
        'median_no_pain':       grp0.median(),
        'median_with_pain':     grp1.median(),
        'U_statistic':          round(u, 1),
        'p_value':              round(p, 4),
        'significant':          p < 0.05
    })

mw_df = pd.DataFrame(mw_results).sort_values('p_value')
mw_df.to_csv(TABLES / 'mann_whitney.csv', index=False)
mw_df

,feature,median_no_pain,median_with_pain,U_statistic,p_value,significant
8,age_ord,0.00,1.000,1228.0,0.0001,True
9,job_duration_ord,0.00,1.000,1261.5,0.0002,True
0,workload_score,41.70,50.000,1291.0,0.0007,True
10,income_ord,1.00,2.000,1410.5,0.0023,True
1,fatigue_score,3.75,4.585,1473.5,0.0078,True
2,force_exertion,3.00,4.000,1585.0,0.0247,True
11,education_ord,2.00,2.000,2606.0,0.0305,True
5,deliveries_num,15.00,25.000,1662.0,0.0413,True
3,vibration_index,9.00,9.000,1945.0,0.4024,False
6,work_days_num,5.50,5.500,1963.0,0.4188,False


## Step 4 — Kruskal-Wallis: numeric predictors across age groups

In [5]:
# Does each numeric predictor differ significantly across age groups?
age_order = ['<25', '25-35', '36-45', '>=46']

kw_results = []
for feat in ['workload_score', 'fatigue_score', 'force_exertion',
             'work_hours_num', 'deliveries_num', 'vibration_index',
             'job_duration_ord', 'income_ord']:
    groups = [df.loc[df['Age'] == g, feat].values for g in age_order]
    h, p = stats.kruskal(*groups)
    kw_results.append({
        'feature':     feat,
        'grouping':    'Age',
        'H_statistic': round(h, 3),
        'p_value':     round(p, 4),
        'significant': p < 0.05
    })

kw_df = pd.DataFrame(kw_results).sort_values('p_value')
kw_df.to_csv(TABLES / 'kruskal_wallis.csv', index=False)
kw_df

,feature,grouping,H_statistic,p_value,significant
6,job_duration_ord,Age,26.538,0.0000,True
7,income_ord,Age,15.456,0.0015,True
1,fatigue_score,Age,11.016,0.0116,True
2,force_exertion,Age,8.046,0.0451,True
0,workload_score,Age,7.058,0.0701,False
4,deliveries_num,Age,3.385,0.3360,False
3,work_hours_num,Age,1.410,0.7032,False
5,vibration_index,Age,0.484,0.9223,False


## Step 5 — Univariable logistic regression: discomfort ~ predictor

In [6]:
# Per-predictor OR + 95% CI for the discomfort outcome
predictors = ['workload_score', 'fatigue_score', 'force_exertion',
              'vibration_index', 'work_hours_num', 'deliveries_num',
              'work_days_num', 'rest_break_num',
              'age_ord', 'job_duration_ord', 'income_ord', 'education_ord',
              'vehicle_rank', 'carrying_contact_rank']

lr_results = []
for p in predictors:
    X = sm.add_constant(df[[p]])
    y = df['discomfort']
    try:
        model = sm.Logit(y, X).fit(disp=0)
        coef = model.params[p]
        ci   = model.conf_int().loc[p]
        lr_results.append({
            'predictor':   p,
            'OR':          round(np.exp(coef), 3),
            'CI_low':      round(np.exp(ci[0]), 3),
            'CI_high':     round(np.exp(ci[1]), 3),
            'p_value':     round(model.pvalues[p], 4),
            'significant': model.pvalues[p] < 0.05
        })
    except Exception as e:
        lr_results.append({'predictor': p, 'OR': np.nan, 'CI_low': np.nan,
                           'CI_high': np.nan, 'p_value': np.nan,
                           'significant': False})

lr_df = pd.DataFrame(lr_results).sort_values('p_value')
lr_df.to_csv(TABLES / 'logistic_regression.csv', index=False)
lr_df

,predictor,OR,CI_low,CI_high,p_value,significant
0,workload_score,1.055,1.024,1.088,0.0005,True
8,age_ord,3.583,1.697,7.564,0.0008,True
9,job_duration_ord,2.886,1.539,5.414,0.0010,True
1,fatigue_score,1.428,1.133,1.800,0.0026,True
10,income_ord,2.003,1.253,3.202,0.0037,True
11,education_ord,0.330,0.116,0.937,0.0373,True
5,deliveries_num,1.051,1.001,1.103,0.0454,True
2,force_exertion,1.183,0.997,1.404,0.0537,False
13,carrying_contact_rank,0.687,0.436,1.081,0.1043,False
6,work_days_num,1.229,0.933,1.619,0.1423,False


## Step 6 — Summary of significant findings

In [7]:
summary_rows = []

# Chi-square significant
for _, r in chi_df[chi_df['significant']].iterrows():
    summary_rows.append({
        'test':    'Chi-square',
        'finding': f"{r['risk_factor']} risk associated with discomfort",
        'effect':  f"chi2={r['chi2']}, dof={r['dof']}",
        'p_value': r['p_value']
    })

# Mann-Whitney significant
for _, r in mw_df[mw_df['significant']].iterrows():
    summary_rows.append({
        'test':    'Mann-Whitney',
        'finding': f"{r['feature']} differs between pain vs no-pain",
        'effect':  f"median {r['median_no_pain']} vs {r['median_with_pain']}",
        'p_value': r['p_value']
    })

# Kruskal-Wallis significant
for _, r in kw_df[kw_df['significant']].iterrows():
    summary_rows.append({
        'test':    'Kruskal-Wallis',
        'finding': f"{r['feature']} differs across {r['grouping']} groups",
        'effect':  f"H={r['H_statistic']}",
        'p_value': r['p_value']
    })

# Logistic regression significant
for _, r in lr_df[lr_df['significant']].iterrows():
    summary_rows.append({
        'test':    'Logistic regression',
        'finding': f"{r['predictor']} -> discomfort",
        'effect':  f"OR={r['OR']} (95% CI {r['CI_low']}-{r['CI_high']})",
        'p_value': r['p_value']
    })

summary = pd.DataFrame(summary_rows).sort_values(['test', 'p_value'])
summary.to_csv(TABLES / 'phase5_summary.csv', index=False)

print(f'Total significant findings: {len(summary)}')
summary

Total significant findings: 21


,test,finding,effect,p_value
0,Chi-square,posture risk associated with discomfort,"chi2=45.665, dof=1",0.0000
1,Chi-square,force risk associated with discomfort,"chi2=6.718, dof=2",0.0348
10,Kruskal-Wallis,job_duration_ord differs across Age groups,H=26.538,0.0000
11,Kruskal-Wallis,income_ord differs across Age groups,H=15.456,0.0015
12,Kruskal-Wallis,fatigue_score differs across Age groups,H=11.016,0.0116
13,Kruskal-Wallis,force_exertion differs across Age groups,H=8.046,0.0451
14,Logistic regression,workload_score -> discomfort,OR=1.055 (95% CI 1.024-1.088),0.0005
15,Logistic regression,age_ord -> discomfort,OR=3.583 (95% CI 1.697-7.564),0.0008
16,Logistic regression,job_duration_ord -> discomfort,OR=2.886 (95% CI 1.539-5.414),0.0010
17,Logistic regression,fatigue_score -> discomfort,OR=1.428 (95% CI 1.133-1.8),0.0026
